In [ ]:
from utils.api_client import fetch_crypto_data
from utils.spark_utils import get_spark
from pyspark.sql.functions import current_timestamp

spark = get_spark()

def fetch_batch():
    data = fetch_crypto_data(["bitcoin", "ethereum"])
    return spark.createDataFrame(data)


def process_batch(_, batch_id):

    df = fetch_batch()

    df = df.withColumn("ingestion_time", current_timestamp())

    df.write.format("delta") \
        .mode("append") \
        .saveAsTable("workspace.cryptoinsight.bronze_stream")


In [ ]:
stream = spark.readStream.format("rate").load()

query = stream.writeStream.foreachBatch(process_batch) \
    .option("checkpointLocation", "/Volumes/workspace/cryptoinsight/bronze_stream_checkpoint/") \
    .start()

query.awaitTermination()